In [18]:
# Configuration
TARGET_COLUMN = 'H78.SI'  # Stock ticker for Viatris

# Step 1: Import Libraries and Install Dependencies
print("\nStep 1: Installing dependencies...")
!pip install --force-reinstall tensorflow==2.16.1 numpy==2.0.2 pandas==2.2.2 scikit-learn==1.5.2 optuna==4.0.0 lightgbm==4.5.0 xgboost==2.1.1 pywt==1.5.0 pykalman==0.9.7 transformers==4.44.2 torch==2.4.1 requests==2.32.3 beautifulsoup4==4.12.3
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # For CUDA debugging
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from google.colab import files
import warnings
warnings.filterwarnings('ignore')
import time
import random
import torch


Step 1: Installing dependencies...
  Using cached tensorflow-2.16.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.3 kB)
  Using cached numpy-2.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached pandas-2.2.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
  Using cached scikit_learn-1.5.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (13 kB)
  Using cached optuna-4.0.0-py3-none-any.whl.metadata (16 kB)
  Using cached lightgbm-4.5.0-py3-none-manylinux_2_28_x86_64.whl.metadata (17 kB)
  Using cached xgboost-2.1.1-py3-none-manylinux_2_28_x86_64.whl.metadata (2.1 kB)
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the re

In [19]:
# Optuna
try:
    import optuna
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True
    print("Optuna version:", optuna.__version__)
except Exception:
    print("Optuna installation failed.")
    !pip install optuna==4.0.0
    import optuna
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True

# TensorFlow
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import Dense, LSTM, Bidirectional, Conv1D, GRU, Dropout, MultiHeadAttention, GlobalAveragePooling1D, Input, Flatten
    from tensorflow.keras.regularizers import l2
    TF_AVAILABLE = True
    print("TensorFlow version:", tf.__version__)
except Exception:
    print("TensorFlow installation failed.")
    !pip install tensorflow==2.16.1
    import tensorflow as tf
    TF_AVAILABLE = True

# PyWavelets
try:
    import pywt
    PYWT_AVAILABLE = True
except Exception:
    print("PyWavelets installation failed.")
    !pip install pywt==1.5.0
    import pywt
    PYWT_AVAILABLE = True

# PyKalman
try:
    from pykalman import KalmanFilter
    PYKALMAN_AVAILABLE = True
except Exception:
    print("PyKalman installation failed.")
    !pip install pykalman==0.9.7
    from pykalman import KalmanFilter
    PYKALMAN_AVAILABLE = True

# LightGBM
try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except Exception:
    LGB_AVAILABLE = False

# XGBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

# Transformers and PyTorch
TRANSFORMERS_AVAILABLE = False
try:
    import transformers
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
    TRANSFORMERS_AVAILABLE = True
    print("Transformers version:", transformers.__version__)
    print("PyTorch version:", torch.__version__)
except Exception:
    print("Transformers or PyTorch installation failed.")
    !pip install transformers==4.44.2 torch==2.4.1
    import transformers
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
    TRANSFORMERS_AVAILABLE = True

# Requests and BeautifulSoup
import requests
from bs4 import BeautifulSoup

Optuna version: 4.0.0
TensorFlow version: 2.18.0
Transformers version: 4.53.0
PyTorch version: 2.6.0+cu124


In [20]:
# Step 2: Load and Clean Data
print("\nStep 2: Loading and cleaning data...")
try:
    uploaded = files.upload()
    stock_data = {}
    news_data = None
    for file_name in uploaded.keys():
        print(f"Processing file: {file_name}")
        if file_name.endswith('.csv'):
            if 'news_data' in file_name.lower():
                try:
                    news_data = pd.read_csv(file_name)
                    print(f"News data loaded: {len(news_data)} rows")
                    print("News data columns:", news_data.columns.tolist())
                    print("News data sample:\n", news_data.head(2))
                except Exception as e:
                    print(f"Error loading news data {file_name}: {e}")
                    news_data = None
            else:
                try:
                    df = pd.read_csv(file_name)
                    print(f"Columns: {df.columns.tolist()}, Rows: {len(df)}")
                    # Check for both 'Timestamp' and 'Timestamp '
                    timestamp_col = 'Timestamp' if 'Timestamp' in df.columns else 'Timestamp ' if 'Timestamp ' in df.columns else None
                    if timestamp_col is None or TARGET_COLUMN not in df.columns:
                        raise ValueError(f"Expected 'Timestamp' or 'Timestamp ' and '{TARGET_COLUMN}' columns")
                    stock_data[TARGET_COLUMN] = df
                    stock_data[TARGET_COLUMN].rename(columns={timestamp_col: 'Timestamp'}, inplace=True) # Rename to 'Timestamp'
                    stock_data[TARGET_COLUMN][TARGET_COLUMN] = stock_data[TARGET_COLUMN][TARGET_COLUMN].interpolate().ffill().bfill()
                    stock_data[TARGET_COLUMN] = stock_data[TARGET_COLUMN].dropna()
                    stock_data[TARGET_COLUMN]['Timestamp'] = pd.to_datetime(stock_data[TARGET_COLUMN]['Timestamp'], errors='coerce')
                    stock_data[TARGET_COLUMN] = stock_data[TARGET_COLUMN].dropna(subset=['Timestamp'])
                    print(f"Rows after cleaning: {len(stock_data[TARGET_COLUMN])}")
                    print("Price data sample:\n", stock_data[TARGET_COLUMN].head(2))
                except Exception as e:
                    print(f"Error loading price data {file_name}: {e}")
except Exception as e:
    print(f"Error during file upload: {e}")
    exit()


Step 2: Loading and cleaning data...


Saving H78.SI_10.csv to H78.SI_10.csv
Processing file: H78.SI_10.csv
Columns: ['Timestamp ', 'H78.SI'], Rows: 6500
Rows after cleaning: 6500
Price data sample:
                   Timestamp  H78.SI
0 2025-05-05 01:00:00+00:00    4.92
1 2025-05-05 01:01:00+00:00    4.93


In [21]:
# Step 2.5: Fetch News from FinViz if No News Data Provided
if news_data is None and TRANSFORMERS_AVAILABLE:
    print("\nStep 2.5: No news data uploaded. Fetching news from FinViz...")
    def fetch_finviz_news(ticker):
        url = f'https://finviz.com/quote.ashx?t={ticker}'
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
        }
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            news_table = soup.find('table', {'id': 'news-table'})
            if not news_table:
                print("No news table found on FinViz")
                return None
            news_data = []
            for row in news_table.findAll('tr'):
                cells = row.findAll('td')
                if len(cells) < 2:
                    continue
                headline_link = cells[1].find('a')
                if not headline_link:
                    continue
                headline = headline_link.text.strip()
                link = headline_link['href']
                if link.startswith('/'):
                    link = f'https://finviz.com{link}'
                news_data.append({'Headline': headline, 'Headline_Link': link})
                time.sleep(0.5)
            df = pd.DataFrame(news_data)
            df = df.head(100)  # Limit to last 100 articles
            df['News_Index'] = range(1, len(df) + 1)  # 1 (least recent) to 100 (most recent)
            print(f"Fetched {len(df)} news articles from FinViz")
            return df
        except Exception as e:
            print(f"Error fetching FinViz data for {ticker}: {e}")
            return None

    def fetch_article_content(url, headline):
        user_agents = [
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36'
        ]
        for attempt in range(3):
            try:
                headers = {'User-Agent': random.choice(user_agents)}
                response = requests.get(url, headers=headers, timeout=10)
                response.raise_for_status()
                soup = BeautifulSoup(response.text, 'html.parser')
                paragraphs = soup.find_all('p')
                article_text = ' '.join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])
                if len(article_text.strip()) > 50:
                    print(f"Successfully fetched article from {url[:50]}...")
                    return article_text
                else:
                    print(f"Short content from {url[:50]}...")
                    return headline
            except Exception as e:
                print(f"Error fetching article from {url[:50]}...: {e}")
                if attempt < 2:
                    time.sleep(2 ** attempt)
                else:
                    return headline
        return headline

    news_data = fetch_finviz_news(TARGET_COLUMN)
    if news_data is not None and not news_data.empty:
        print(f"Fetching article content for {len(news_data)} articles...")
        news_data['News_detail'] = news_data.apply(lambda row: fetch_article_content(row['Headline_Link'], row['Headline']), axis=1)
        news_data['News_detail'] = news_data['News_detail'].apply(lambda x: str(x) if x and len(str(x).strip()) > 0 else None)
        valid_articles = news_data['News_detail'].notna().sum()
        print(f"Fetched {valid_articles} valid articles out of {len(news_data)}")
        if valid_articles == 0:
            print("No valid article content fetched. Disabling sentiment analysis.")
            news_data = None
    else:
        print("Failed to fetch news from FinViz.")


Step 2.5: No news data uploaded. Fetching news from FinViz...
Error fetching FinViz data for H78.SI: 404 Client Error: Not Found for url: https://finviz.com/quote.ashx?t=H78.SI
Failed to fetch news from FinViz.


In [22]:
# Step 2.6: Process News Data with FinBERT
if TRANSFORMERS_AVAILABLE and news_data is not None:
    print("\nStep 2.6: Processing news data with FinBERT...")
    tokenizer = None
    model = None
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU if available
    try:
        torch.cuda.empty_cache()
        tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
        model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
        model.to(device)
        print(f"FinBERT model loaded on {device}.")
    except Exception as e:
        print(f"Failed to load FinBERT: {e}")
        TRANSFORMERS_AVAILABLE = False
        news_data = None

    if news_data is not None:
        print("Summarizing news articles...")
        summarizer = None
        try:
            torch.cuda.empty_cache()
            # Using device=-1 for CPU, which is the default for summarization pipeline
            summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=-1)
            print("Summarizer loaded on CPU.")
        except Exception as e:
            print(f"Failed to load summarizer: {e}")
            # Fallback to using headline if summarizer fails
            news_data['Summary'] = news_data['News_detail'].fillna(news_data['Headline'])

        if summarizer:
            def summarize_text(text):
                if not text or pd.isna(text) or len(str(text).strip()) == 0:
                    return ""
                text = str(text)[:1024]  # Truncate to avoid exceeding max length
                word_count = len(str(text).split())
                if word_count <= 30:  # Summarize only if text long enough
                    return text
                try:
                    # Adjust max_length and min_length based on input length
                    max_len = min(150, max(30, int(word_count * 0.3)))
                    min_len = max(10, min(max_len, int(word_count * 0.1)))
                    summary = summarizer(text, max_length=max_len, min_length=min_len, do_sample=False, batch_size=1)[0]['summary_text']
                    return summary
                except Exception as e:
                    print(f"Error in summarization for text {text[:50]}...: {e}")
                    return text[:256]  # Return truncated text on failure

            # Apply summarization only to rows where News_detail is not None and not empty
            news_data['Summary'] = news_data.apply(
                lambda row: summarize_text(row['News_detail']) if pd.notna(row['News_detail']) and len(str(row['News_detail']).strip()) > 0 else row['Headline'],
                axis=1
            )

            unique_summaries = len(news_data['Summary'].unique())
            print(f"\nGenerated {unique_summaries} unique summaries")
            print(f"News summaries:\n", news_data[['Summary']].head(5))

        def get_finbert_sentiment(text):
            if not text or pd.isna(text) or len(str(text).strip()) == 0:
                return np.array([0.33, 0.33, 0.33])  # Return neutral default scores for invalid text
            try:
                # Use the text directly for sentiment analysis, truncation handled by tokenizer
                inputs = tokenizer(str(text), return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
                with torch.no_grad():
                    outputs = model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
                return probs.cpu().numpy().flatten()
            except Exception as e:
                print(f"Error analyzing sentiment for text {str(text)[:50]}...: {e}")
                return np.array([0.33, 0.33, 0.33])  # Return neutral scores on failure

        # Apply sentiment analysis to the 'Summary' column
        news_data['sentiment_scores'] = news_data['Summary'].apply(
            lambda x: get_finbert_sentiment(x)
        )

        # Expand sentiment scores into separate columns
        sentiment_scores_list = news_data['sentiment_scores'].tolist()
        sentiment_scores_df = pd.DataFrame(sentiment_scores_list, index=news_data.index)
        news_data[['positive', 'negative', 'neutral']] = sentiment_scores_df

        news_data['news_polarity'] = news_data['positive'] - news_data['negative']
        news_data['news_strength'] = news_data[['positive', 'negative', 'neutral']].max(axis=1)
        news_data['news_weight'] = news_data['News_Index'] / news_data['News_Index'].max() if news_data['News_Index'].max() > 0 else 0  # Normalize weight
        news_data['news_impact'] = news_data['news_polarity'] * news_data['news_strength'] * news_data['news_weight']
        print(f"Sentiment polarity and impact:\n", news_data[['News_Index', 'positive', 'negative', 'neutral', 'news_polarity', 'news_strength', 'news_weight', 'news_impact']].head())

        # Merge news impact data with price data
        try:
            if TARGET_COLUMN in stock_data and len(stock_data[TARGET_COLUMN]) > 0:
                print(f"Aligning news data with price data...")
                stock_df = stock_data[TARGET_COLUMN].sort_values(by='Timestamp').reset_index(drop=True)
                news_data_aligned = news_data.reset_index(drop=True)  # Ensure index is reset

                # Ensure news_data_aligned has same number of rows as stock_df for direct assignment
                if len(news_data_aligned) > len(stock_df):
                    news_data_aligned = news_data_aligned.iloc[:len(stock_df)]
                elif len(news_data_aligned) < len(stock_df):
                    # If news data is shorter, replicate the last row to match the length
                    last_row = news_data_aligned.iloc[-1]
                    filler_rows = pd.DataFrame([last_row] * (len(stock_df) - len(news_data_aligned)))
                    news_data_aligned = pd.concat([news_data_aligned, filler_rows], ignore_index=True)

                # Add news sentiment columns to stock_df
                news_sentiment_cols = ['positive', 'negative', 'neutral', 'news_polarity', 'news_strength', 'news_weight', 'news_impact']
                if all(col in news_data_aligned.columns for col in news_sentiment_cols):
                    merged_data = stock_df.copy()
                    # Assign the sentiment columns directly as both dataframes have same length and index
                    merged_data[news_sentiment_cols] = news_data_aligned[news_sentiment_cols]
                    # Fill potential NaN with neutral sentiment
                    merged_data[news_sentiment_cols] = merged_data[news_sentiment_cols].fillna(0.33)  # Assuming 0.33 is neutral for probability-based scores
                    stock_data[TARGET_COLUMN] = merged_data
                    print(f"Merged data shape: {stock_data[TARGET_COLUMN].shape}")
                    print(f"Merged sample:\n", stock_data[TARGET_COLUMN].head(5))
                else:
                    print("Sentiment columns not found in news_data after alignment. Proceeding without sentiment features.")
            else:
                print(f"No valid stock data or empty. Exiting...")
                exit()
        except Exception as e:
            print(f"Error during merging news data: {e}")
            print("Proceeding without sentiment features...")
            news_data = None
else:
    print("No news_data available or Transformers unavailable.")

No news_data available or Transformers unavailable.


In [23]:
# Step 3: Denoise Data for Highest SNR
print("\nStep 3: Denoising data...")
price_scaler = MinMaxScaler()
feature_columns = [TARGET_COLUMN]
if TRANSFORMERS_AVAILABLE and news_data is not None:
    feature_columns.extend(['positive', 'negative', 'neutral', 'news_polarity', 'news_strength', 'news_impact'])

stock_denoised_data = {} # Initialize stock_denoised_data dictionary

if TARGET_COLUMN in stock_data and not stock_data[TARGET_COLUMN].empty:
    scaled_data = price_scaler.fit_transform(stock_data[TARGET_COLUMN][[TARGET_COLUMN]])
    print(f"Scaled data shape: {scaled_data.shape}")

    def calculate_snr(original, denoised):
        noise = original - denoised
        signal_power = np.mean(original ** 2)
        noise_power = np.var(noise)
        return 10 * np.log10(signal_power / noise_power) if noise_power > 0 else float('inf')

    def optimize_denoising(trial, data):
        methods = []
        if PYWT_AVAILABLE:
            wavelet = trial.suggest_categorical('wavelet', ['db1', 'db2', 'sym2'])
            level = trial.suggest_int('wavelet_level', 1, 3)
            wavelet_denoised = pywt.wavedec(data, wavelet, level=level)
            threshold = np.std(wavelet_denoised[-1]) * np.sqrt(2 * np.log(len(data)))
            wavelet_denoised = pywt.waverec([pywt.threshold(c, threshold, mode='soft') for c in wavelet_denoised], wavelet)[:len(data)]
            snr_wavelet = calculate_snr(data, wavelet_denoised)
            methods.append(('wavelet', snr_wavelet, wavelet_denoised))
        ma_window = trial.suggest_int('ma_window', 3, 10)
        ma_denoised = pd.Series(data).rolling(window=ma_window, min_periods=1).mean().values
        snr_ma = calculate_snr(data, ma_denoised)
        methods.append(('ma', snr_ma, ma_denoised))
        if TF_AVAILABLE:
            encoding_dim = trial.suggest_int('encoding_dim', 16, 32)
            model = Sequential([Dense(encoding_dim, activation='relu', input_shape=(1,)),
                                Dense(1, activation='linear')])
            model.compile(optimizer='adam', loss='mse')
            autoencoder_denoised = model.predict(data.reshape(-1, 1), verbose=0).flatten()
            snr_autoencoder = calculate_snr(data, autoencoder_denoised)
            methods.append(('autoencoder', snr_autoencoder, autoencoder_denoised))
        if PYKALMAN_AVAILABLE:
            kf = KalmanFilter(transition_matrices=[1], observation_matrices=[1], initial_state_mean=data[0])
            kalman_denoised, _ = kf.filter(data)
            snr_kalman = calculate_snr(data, kalman_denoised.flatten())
            methods.append(('kalman', snr_kalman, kalman_denoised))
        best_method = max(methods, key=lambda x: x[1] if not np.isinf(x[1]) else -float('inf'))
        return best_method[1], best_method[2]

    if OPTUNA_AVAILABLE:
        print("Optimizing denoising with Optuna...")
        study_denoise = optuna.create_study(direction='maximize', sampler=TPESampler())
        study_denoise.optimize(lambda trial: optimize_denoising(trial, scaled_data[:, 0])[0], n_trials=5)
        _, denoised_data = optimize_denoising(optuna.trial.FixedTrial(study_denoise.best_params), scaled_data[:, 0])
        print(f"Best SNR after Optuna: {calculate_snr(scaled_data[:, 0], denoised_data):.2f} dB")
    else:
        print("Optuna unavailable. Using default moving average denoising...")
        denoised_data = pd.Series(scaled_data[:, 0]).rolling(window=5, min_periods=1).mean().values
        print(f"Best SNR (default): {calculate_snr(scaled_data[:, 0], denoised_data):.2f} dB")

    # Ensure stock_denoised_data is populated
    stock_denoised_data[TARGET_COLUMN] = stock_data[TARGET_COLUMN].copy()
    stock_denoised_data[TARGET_COLUMN][TARGET_COLUMN] = denoised_data

    if TRANSFORMERS_AVAILABLE and news_data is not None and all(col in news_data.columns for col in ['positive', 'negative', 'neutral', 'news_polarity', 'news_strength', 'news_impact']):
         # Align and merge news sentiment data with denoised stock data
        news_sentiment_cols = ['positive', 'negative', 'neutral', 'news_polarity', 'news_strength', 'news_impact']
        # Ensure news_data_aligned has same number of rows as stock_data[TARGET_COLUMN] for direct assignment
        if len(news_data) > len(stock_data[TARGET_COLUMN]):
            news_data_aligned = news_data.iloc[:len(stock_data[TARGET_COLUMN])].reset_index(drop=True)
        elif len(news_data) < len(stock_data[TARGET_COLUMN]):
            last_row = news_data.iloc[-1]
            filler_rows = pd.DataFrame([last_row] * (len(stock_data[TARGET_COLUMN]) - len(news_data))).reset_index(drop=True)
            news_data_aligned = pd.concat([news_data.reset_index(drop=True), filler_rows], ignore_index=True)
        else:
            news_data_aligned = news_data.reset_index(drop=True)

        stock_denoised_data[TARGET_COLUMN][news_sentiment_cols] = news_data_aligned[news_sentiment_cols]
        stock_denoised_data[TARGET_COLUMN][news_sentiment_cols] = stock_denoised_data[TARGET_COLUMN][news_sentiment_cols].fillna(0.33) # Fill NaN with neutral

    print(f"Stock denoised data shape: {stock_denoised_data[TARGET_COLUMN].shape}")
    print(f"Stock denoised data sample:\n", stock_denoised_data[TARGET_COLUMN].head(5))

else:
    print("No valid stock data available after cleaning. Cannot proceed with denoising.")
    # Ensure stock_denoised_data is still an empty dict if no data
    stock_denoised_data = {}

[I 2025-07-06 18:09:49,664] A new study created in memory with name: no-name-cfbee2ac-34c5-42ca-9214-ca7e661cccb3



Step 3: Denoising data...
Scaled data shape: (6500, 1)
Optimizing denoising with Optuna...


[I 2025-07-06 18:09:51,982] Trial 0 finished with value: 41.06797051032604 and parameters: {'wavelet': 'db1', 'wavelet_level': 1, 'ma_window': 8, 'encoding_dim': 30}. Best is trial 0 with value: 41.06797051032604.
[I 2025-07-06 18:09:53,908] Trial 1 finished with value: 41.06797051032604 and parameters: {'wavelet': 'db1', 'wavelet_level': 2, 'ma_window': 4, 'encoding_dim': 23}. Best is trial 0 with value: 41.06797051032604.
[I 2025-07-06 18:09:55,704] Trial 2 finished with value: 41.06797051032604 and parameters: {'wavelet': 'sym2', 'wavelet_level': 3, 'ma_window': 8, 'encoding_dim': 32}. Best is trial 0 with value: 41.06797051032604.
[I 2025-07-06 18:09:57,398] Trial 3 finished with value: 41.06797051032604 and parameters: {'wavelet': 'db1', 'wavelet_level': 2, 'ma_window': 10, 'encoding_dim': 21}. Best is trial 0 with value: 41.06797051032604.
[I 2025-07-06 18:09:59,210] Trial 4 finished with value: 41.06797051032604 and parameters: {'wavelet': 'sym2', 'wavelet_level': 3, 'ma_window'

Best SNR after Optuna: 6.28 dB
Stock denoised data shape: (6500, 2)
Stock denoised data sample:
                   Timestamp    H78.SI
0 2025-05-05 01:00:00+00:00  0.136363
1 2025-05-05 01:01:00+00:00  0.150000
2 2025-05-05 01:02:00+00:00  0.169580
3 2025-05-05 01:03:00+00:00  0.163101
4 2025-05-05 01:04:00+00:00  0.132533


In [24]:
# Step 4: Prepare Sequences
print("\nStep 4: Preparing sequences...")
SEQ_LENGTH = 30
WINDOW_SIZE = 300
stock_X_deep = {}
stock_X_ml = {}
stock_y = {}
full_data = stock_denoised_data[TARGET_COLUMN][feature_columns].values
X_for_prediction = np.array([full_data[i:i + SEQ_LENGTH] for i in range(len(full_data) - SEQ_LENGTH)])
stock_y[TARGET_COLUMN] = stock_denoised_data[TARGET_COLUMN][TARGET_COLUMN].values[SEQ_LENGTH:]
stock_X_deep[TARGET_COLUMN] = X_for_prediction
stock_X_ml[TARGET_COLUMN] = X_for_prediction
print(f"X_deep shape: {stock_X_deep[TARGET_COLUMN].shape}, y shape: {stock_y[TARGET_COLUMN].shape}")
last_300_scaled = stock_denoised_data[TARGET_COLUMN][feature_columns].values[-WINDOW_SIZE:]


Step 4: Preparing sequences...
X_deep shape: (6470, 30, 1), y shape: (6470,)


In [25]:
# Step 5: Define and Optimize Models
print("\nStep 5: Defining and optimizing models...")
feature_dim = len(feature_columns)
def optimize_cnn_lstm(trial):
    filters = trial.suggest_int('filters', 32, 64)
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Conv1D(filters=filters, kernel_size=3, padding='same', activation='relu', kernel_regularizer=l2(0.02)),
        LSTM(lstm_units, return_sequences=True, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        LSTM(lstm_units // 2, kernel_regularizer=l2(0.02)),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_conv1d_lstm(trial):
    filters = trial.suggest_int('filters', 32, 64)
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Conv1D(filters=filters, kernel_size=3, padding='same', activation='relu', kernel_regularizer=l2(0.02)),
        LSTM(lstm_units, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_bilstm(trial):
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Bidirectional(LSTM(lstm_units, kernel_regularizer=l2(0.02))),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_stacked_lstm(trial):
    lstm_units = trial.suggest_int('lstm_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        LSTM(lstm_units, return_sequences=True, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        LSTM(lstm_units // 2, kernel_regularizer=l2(0.02)),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_gru(trial):
    gru_units = trial.suggest_int('gru_units', 50, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        GRU(gru_units, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_lightgbm(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-5, 1.0)
    }
    return lgb.LGBMRegressor(**params, force_col_wise=True)

def optimize_xgboost(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0)
    }
    return xgb.XGBRegressor(**params)

def optimize_transformer(trial):
    num_heads = trial.suggest_int('num_heads', 4, 8)
    ff_dim = trial.suggest_int('ff_dim', 64, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    inputs = Input(shape=(SEQ_LENGTH, feature_dim))
    x = MultiHeadAttention(num_heads=num_heads, key_dim=SEQ_LENGTH // num_heads)(inputs, inputs)
    x = Dropout(dropout_rate)(x)
    x = MultiHeadAttention(num_heads=num_heads, key_dim=SEQ_LENGTH // num_heads)(x, x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(ff_dim, activation='relu', kernel_regularizer=l2(0.02))(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_tcn(trial):
    filters = trial.suggest_int('filters', 32, 64)
    kernel_size = trial.suggest_int('kernel_size', 2, 5)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.6)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3)
    inputs = Input(shape=(SEQ_LENGTH, feature_dim))
    x = inputs
    shortcut = x
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu', kernel_regularizer=l2(0.02))(x)
    x = Dropout(dropout_rate)(x)
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu', kernel_regularizer=l2(0.02))(x)
    shortcut = Conv1D(filters=filters, kernel_size=1, padding='same')(shortcut)
    x = tf.keras.layers.Add()([shortcut, x])
    x = tf.keras.layers.Activation('relu')(x)
    x = Flatten()(x)
    x = Dense(32, activation='relu', kernel_regularizer=l2(0.02))(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model


Step 5: Defining and optimizing models...


In [26]:
models = [
    ('cnn_lstm', optimize_cnn_lstm, 'deep'),
    ('conv1d_lstm', optimize_conv1d_lstm, 'deep'),
    ('bilstm', optimize_bilstm, 'deep'),
    ('stacked_lstm', optimize_stacked_lstm, 'deep'),
    ('gru', optimize_gru, 'deep'),
    ('lightgbm', optimize_lightgbm, 'ml'),
    ('xgboost', optimize_xgboost, 'ml'),
    ('transformer', optimize_transformer, 'deep'),
    ('tcn', optimize_tcn, 'deep')
]

# Step 6: Train and Evaluate Models with Walk-Forward Validation
print("\nStep 6: Training and evaluating models...")
def walk_forward_validation(model, model_name, X, y, price_scaler, step_size=300, max_steps=20):
    best_adj_r2 = -float('inf')
    best_model = None
    adj_r2_history = []
    n_steps = min(max_steps, max(1, (len(X) - SEQ_LENGTH) // step_size))
    if n_steps < 1:
        print(f"Error: Insufficient data for walk-forward validation for {model_name}.")
        return {'adj_r2': 0, 'best_model': model}, adj_r2_history
    n_predictors = feature_dim
    for i in range(1, n_steps + 1):
        train_end = i * step_size
        test_size = min(step_size, len(X) - train_end)
        train_X = X[:train_end]
        train_y = y[:train_end]
        test_X = X[train_end:train_end + test_size]
        test_y = y[train_end:train_end + test_size]
        if len(test_X) == 0 or len(test_y) == 0 or len(test_X) != len(test_y):
            continue
        try:
            if model_name in ['xgboost', 'lightgbm']:
                model.fit(train_X[:, :, 0].reshape(-1, SEQ_LENGTH), train_y)
                preds = model.predict(test_X[:, :, 0].reshape(-1, SEQ_LENGTH))
            else:
                model.fit(train_X, train_y, epochs=10, batch_size=64, validation_split=0.2, verbose=0,
                          callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
                preds = model.predict(test_X, verbose=0).flatten()
            test_y_inv = price_scaler.inverse_transform(test_y.reshape(-1, 1)).flatten()
            preds_inv = price_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
            ss_res = np.sum((test_y_inv - preds_inv) ** 2)
            ss_tot = np.sum((test_y_inv - np.mean(test_y_inv)) ** 2)
            r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
            n = len(test_y_inv)
            adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_predictors - 1) if n > n_predictors + 1 else 0
            adj_r2_history.append(adj_r2)
            if adj_r2 > best_adj_r2:
                best_adj_r2 = adj_r2
                best_model = tf.keras.models.clone_model(model) if model_name not in ['xgboost', 'lightgbm'] else model
                if model_name not in ['xgboost', 'lightgbm']:
                    best_model.set_weights(model.get_weights())
        except Exception as e:
            print(f"Error in walk-forward validation for {model_name} at step {i}: {e}")
            continue
    return {'adj_r2': best_adj_r2, 'best_model': best_model}, adj_r2_history


Step 6: Training and evaluating models...


In [27]:
stock_results = {}
for stock_name in stock_data.keys():
    results = {}
    for model_name, model_fn, data_type in models:
        if not (TF_AVAILABLE and data_type == 'deep') and not (LGB_AVAILABLE and model_name == 'lightgbm') and not (XGB_AVAILABLE and model_name == 'xgboost'):
            continue
        print(f"\nTraining {model_name} for {stock_name}...")
        X_data = stock_X_deep[stock_name] if data_type == 'deep' else stock_X_ml[stock_name]
        try:
            if OPTUNA_AVAILABLE:
                study = optuna.create_study(direction='minimize', sampler=TPESampler())
                def objective(trial):
                    model = model_fn(trial)
                    n_samples = len(X_data)
                    train_size = int(0.8 * n_samples)
                    train_X, train_y = X_data[:train_size], stock_y[stock_name][:train_size]
                    val_X, val_y = X_data[train_size:], stock_y[stock_name][train_size:]
                    if model_name in ['xgboost', 'lightgbm']:
                        model.fit(train_X[:, :, 0].reshape(-1, SEQ_LENGTH), train_y)
                        preds = model.predict(val_X[:, :, 0].reshape(-1, SEQ_LENGTH))
                    else:
                        model.fit(train_X, train_y, epochs=5, batch_size=64, validation_split=0.2, verbose=0,
                                  callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)])
                        preds = model.predict(val_X, verbose=0).flatten()
                    val_y_inv = price_scaler.inverse_transform(val_y.reshape(-1, 1)).flatten()
                    preds_inv = price_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
                    ss_res = np.sum((val_y_inv - preds_inv) ** 2)
                    ss_tot = np.sum((val_y_inv - np.mean(val_y_inv)) ** 2)
                    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
                    n = len(val_y_inv)
                    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - feature_dim - 1) if n > feature_dim + 1 else 0
                    return -adj_r2
                study.optimize(objective, n_trials=3)
                model = model_fn(optuna.trial.FixedTrial(study.best_params))
            else:
                model = model_fn({})
            result, adj_r2_history = walk_forward_validation(model, model_name, X_data, stock_y[stock_name], price_scaler)
            results[model_name] = result
            print(f"{model_name} - {stock_name} - Best Adj R²: {result['adj_r2']:.6f}")
        except Exception as e:
            print(f"Error training {model_name} for {stock_name}: {e}")
            results[model_name] = {'adj_r2': 0, 'best_model': None}
    stock_results[stock_name] = results

[I 2025-07-06 18:10:47,590] A new study created in memory with name: no-name-c9d71f5e-d5ea-41ca-926d-6ffbaa78c1fe



Training cnn_lstm for H78.SI...


[I 2025-07-06 18:11:13,120] Trial 0 finished with value: -0.1879202204080932 and parameters: {'filters': 64, 'lstm_units': 78, 'dropout_rate': 0.559427127273054, 'learning_rate': 0.003828224106156902}. Best is trial 0 with value: -0.1879202204080932.
[I 2025-07-06 18:11:22,873] Trial 1 finished with value: -0.7496110502376473 and parameters: {'filters': 41, 'lstm_units': 88, 'dropout_rate': 0.5873430024698896, 'learning_rate': 0.004260557062905251}. Best is trial 1 with value: -0.7496110502376473.
[I 2025-07-06 18:11:34,506] Trial 2 finished with value: -0.19076965177820449 and parameters: {'filters': 39, 'lstm_units': 72, 'dropout_rate': 0.544444999805145, 'learning_rate': 0.004800063900251263}. Best is trial 1 with value: -0.7496110502376473.
[I 2025-07-06 18:14:37,371] A new study created in memory with name: no-name-06528615-22e1-4b90-b57d-a1e3bf53b2a3


cnn_lstm - H78.SI - Best Adj R²: -3.693549

Training conv1d_lstm for H78.SI...


[I 2025-07-06 18:14:45,621] Trial 0 finished with value: 0.7565548316143045 and parameters: {'filters': 54, 'lstm_units': 123, 'dropout_rate': 0.369065413132913, 'learning_rate': 0.003987556885382407}. Best is trial 0 with value: 0.7565548316143045.
[I 2025-07-06 18:14:53,495] Trial 1 finished with value: -0.5679156603699285 and parameters: {'filters': 34, 'lstm_units': 94, 'dropout_rate': 0.5470751327757292, 'learning_rate': 0.003284947862658046}. Best is trial 1 with value: -0.5679156603699285.
[I 2025-07-06 18:15:00,715] Trial 2 finished with value: -0.19814197314570037 and parameters: {'filters': 40, 'lstm_units': 70, 'dropout_rate': 0.3648093824517052, 'learning_rate': 0.003435714388390163}. Best is trial 1 with value: -0.5679156603699285.
[I 2025-07-06 18:17:09,915] A new study created in memory with name: no-name-c9edf685-ba7b-4864-a1b8-af9008a4ca15


...

xgboost - H78.SI - Best Adj R²: 0.977405

Training transformer for H78.SI...


[I 2025-07-06 18:27:41,483] Trial 0 finished with value: 4.427379272122222 and parameters: {'num_heads': 8, 'ff_dim': 113, 'dropout_rate': 0.5007514272281364, 'learning_rate': 0.00433445363934202}. Best is trial 0 with value: 4.427379272122222.
[I 2025-07-06 18:27:55,033] Trial 1 finished with value: 7.137813209527772 and parameters: {'num_heads': 6, 'ff_dim': 92, 'dropout_rate': 0.5701461590246044, 'learning_rate': 0.003205111587741913}. Best is trial 0 with value: 4.427379272122222.
[I 2025-07-06 18:28:10,820] Trial 2 finished with value: -0.8290825594120088 and parameters: {'num_heads': 5, 'ff_dim': 107, 'dropout_rate': 0.5931718533656156, 'learning_rate': 0.004330565652723787}. Best is trial 2 with value: -0.8290825594120088.
[I 2025-07-06 18:29:28,884] A new study created in memory with name: no-name-ac9b4d7e-6871-48e6-bf89-924ce7759c11


transformer - H78.SI - Best Adj R²: 0.827154

Training tcn for H78.SI...


[I 2025-07-06 18:29:43,244] Trial 0 finished with value: -0.8321558156552349 and parameters: {'filters': 35, 'kernel_size': 2, 'dropout_rate': 0.5177354127810999, 'learning_rate': 0.001988302199463628}. Best is trial 0 with value: -0.8321558156552349.
[I 2025-07-06 18:29:53,206] Trial 1 finished with value: -0.8748005425002778 and parameters: {'filters': 42, 'kernel_size': 5, 'dropout_rate': 0.4504406318152052, 'learning_rate': 0.0020402791768065935}. Best is trial 1 with value: -0.8748005425002778.
[I 2025-07-06 18:30:07,538] Trial 2 finished with value: -0.8553211784003489 and parameters: {'filters': 62, 'kernel_size': 4, 'dropout_rate': 0.5570055705110832, 'learning_rate': 0.002824012850982895}. Best is trial 1 with value: -0.8748005425002778.


tcn - H78.SI - Best Adj R²: 0.929893


In [28]:
# Step 6.5: Analyze Overfitting with Validation and Test Sets
print("\nStep 6.5: Analyzing overfitting with validation and test sets...")
for stock_name in stock_results:
    results = stock_results[stock_name]
    X_data = stock_X_deep[stock_name] if 'cnn_lstm' in results else stock_X_ml[stock_name]
    y_data = stock_y[stock_name]
    n_samples = len(X_data)

    # Split data: last 150 for test, remaining for train+val
    test_size = 150
    if n_samples <= test_size:
        print(f"Error: Not enough samples for {stock_name}. Need > {test_size} samples, got {n_samples}")
        continue
    train_val_size = n_samples - test_size
    train_size = int(0.8 * train_val_size)  # 80% of train+val for training
    val_size = train_val_size - train_size

    train_X, val_X, test_X = X_data[:train_size], X_data[train_size:train_val_size], X_data[train_val_size:]
    train_y, val_y, test_y = y_data[:train_size], y_data[train_size:train_val_size], y_data[train_val_size:]

    for model_name, result in results.items():
        best_model = result['best_model']
        if best_model is None:
            print(f"No model found for {model_name} in {stock_name}")
            continue
        try:
            if model_name in ['xgboost', 'lightgbm']:
                train_pred = best_model.predict(train_X[:, :, 0].reshape(-1, SEQ_LENGTH))
                val_pred = best_model.predict(val_X[:, :, 0].reshape(-1, SEQ_LENGTH))
                test_pred = best_model.predict(test_X[:, :, 0].reshape(-1, SEQ_LENGTH))
            else:
                best_model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse')
                best_model.fit(train_X, train_y, epochs=5, batch_size=64, verbose=0,
                              callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)])
                train_pred = best_model.predict(train_X, verbose=0).flatten()
                val_pred = best_model.predict(val_X, verbose=0).flatten()
                test_pred = best_model.predict(test_X, verbose=0).flatten()

            train_y_inv = price_scaler.inverse_transform(train_y.reshape(-1, 1)).flatten()
            val_y_inv = price_scaler.inverse_transform(val_y.reshape(-1, 1)).flatten()
            test_y_inv = price_scaler.inverse_transform(test_y.reshape(-1, 1)).flatten()
            train_pred_inv = price_scaler.inverse_transform(train_pred.reshape(-1, 1)).flatten()
            val_pred_inv = price_scaler.inverse_transform(val_pred.reshape(-1, 1)).flatten()
            test_pred_inv = price_scaler.inverse_transform(test_pred.reshape(-1, 1)).flatten()

            def calculate_adj_r2(y_true, y_pred):
                ss_res = np.sum((y_true - y_pred) ** 2)
                ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
                r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
                n = len(y_true)
                adj_r2 = 1 - (1 - r2) * (n - 1) / (n - feature_dim - 1) if n > feature_dim + 1 else 0
                return adj_r2

            train_adj_r2 = calculate_adj_r2(train_y_inv, train_pred_inv)
            val_adj_r2 = calculate_adj_r2(val_y_inv, val_pred_inv)
            test_adj_r2 = calculate_adj_r2(test_y_inv, test_pred_inv)
            overfitting_gap = train_adj_r2 - val_adj_r2

            stock_results[stock_name][model_name]['val_adj_r2'] = val_adj_r2
            stock_results[stock_name][model_name]['test_adj_r2'] = test_adj_r2
            stock_results[stock_name][model_name]['overfitting_gap'] = overfitting_gap
            stock_results[stock_name][model_name]['test_predictions'] = test_pred_inv  # Store predictions
            print(f"\nModel: {model_name} for {stock_name} - Val Adj R²: {val_adj_r2:.6f}, Test Adj R²: {test_adj_r2:.6f}, Overfitting Gap: {overfitting_gap:.6f}")
            print(f"  Training Adj R²: {train_adj_r2:.6f}")
            if overfitting_gap > 0.2:
                print("  Warning: Overfitting detected")
            elif overfitting_gap > 0.1:
                print("  Caution: Moderate overfitting risk")
            else:
                print("  Status: Low overfitting risk")
        except Exception as e:
            print(f"Error analyzing overfitting for {model_name} in {stock_name}: {e}")


Step 6.5: Analyzing overfitting with validation and test sets...

Model: cnn_lstm for H78.SI - Val Adj R²: 0.342436, Test Adj R²: -3.477576, Overfitting Gap: 0.601063
  Training Adj R²: 0.943499

Model: conv1d_lstm for H78.SI - Val Adj R²: 0.945784, Test Adj R²: 0.610501, Overfitting Gap: 0.046588
  Training Adj R²: 0.992372
  Status: Low overfitting risk

Model: bilstm for H78.SI - Val Adj R²: 0.962524, Test Adj R²: 0.719202, Overfitting Gap: 0.032475
  Training Adj R²: 0.994999
  Status: Low overfitting risk

Model: stacked_lstm for H78.SI - Val Adj R²: 0.677785, Test Adj R²: -1.288454, Overfitting Gap: 0.294134
  Training Adj R²: 0.971919

Model: gru for H78.SI - Val Adj R²: 0.927753, Test Adj R²: 0.264627, Overfitting Gap: 0.066215
  Training Adj R²: 0.993968
  Status: Low overfitting risk

Model: lightgbm for H78.SI - Val Adj R²: 0.989973, Test Adj R²: 0.820278, Overfitting Gap: 0.008963
  Training Adj R²: 0.998936
  Status: Low overfitting risk

Model: xgboost for H78.SI - Val A

In [29]:
# Step 7: Evaluate Best Model on Test Set
print("\nStep 7: Evaluating best model on test set...")
best_test_adj_r2 = -float('inf')
best_model_name = None
best_model = None
best_stock_name = None

for stock_name in stock_results.keys():
    results = stock_results[stock_name]
    for model_name, result in results.items():
        if result.get('test_adj_r2', -float('inf')) > best_test_adj_r2 and result.get('overfitting_gap', float('inf')) <= 0.2:
            best_test_adj_r2 = result['test_adj_r2']
            best_model_name = model_name
            best_model = result['best_model']
            best_stock_name = stock_name

if best_model_name is None:
    print("No model with acceptable overfitting gap found. Using best Test Adj R².")
    for stock_name in stock_results.keys():
        results = stock_results[stock_name]
        for model_name, result in results.items():
            if result.get('test_adj_r2', -float('inf')) > best_test_adj_r2:
                best_test_adj_r2 = result['test_adj_r2']
                best_model_name = model_name
                best_model = result['best_model']
                best_stock_name = stock_name

overfitting_gap = stock_results[best_stock_name][best_model_name].get('overfitting_gap', 'N/A')
print(f"Best model: {best_model_name} for {best_stock_name} with Test Adj R²: {best_test_adj_r2:.6f}, Overfitting Gap: {overfitting_gap}")

def evaluate_test_set(model, model_name, test_X, test_y, price_scaler, stock_name, timestamps):
    try:
        if model_name in ['xgboost', 'lightgbm']:
            test_pred = model.predict(test_X[:, :, 0].reshape(-1, SEQ_LENGTH))
        else:
            test_pred = model.predict(test_X, verbose=0).flatten()

        test_y_inv = price_scaler.inverse_transform(test_y.reshape(-1, 1)).flatten()
        test_pred_inv = price_scaler.inverse_transform(test_pred.reshape(-1, 1)).flatten()

        def calculate_adj_r2(y_true, y_pred):
            ss_res = np.sum((y_true - y_pred) ** 2)
            ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
            r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
            n = len(y_true)
            adj_r2 = 1 - (1 - r2) * (n - 1) / (n - feature_dim - 1) if n > feature_dim + 1 else 0
            return adj_r2

        test_adj_r2 = calculate_adj_r2(test_y_inv, test_pred_inv)
        test_timestamps = timestamps.iloc[-len(test_y):]
        return test_pred_inv, test_y_inv, test_timestamps, test_adj_r2
    except Exception as e:
        print(f"Prediction failed for {model_name} in {stock_name}: {e}")
        return None, None, None, None


Step 7: Evaluating best model on test set...
Best model: xgboost for H78.SI with Test Adj R²: 0.823853, Overfitting Gap: 0.01446259480825085


In [30]:
# Evaluate best model on test set
test_X = stock_X_deep[best_stock_name][-150:] if best_model_name in ['cnn_lstm', 'conv1d_lstm', 'bilstm', 'stacked_lstm', 'gru', 'transformer', 'tcn'] else stock_X_ml[best_stock_name][-150:]
test_y = stock_y[best_stock_name][-150:]
predictions, actuals, test_timestamps, test_adj_r2 = evaluate_test_set(
    best_model, best_model_name, test_X, test_y, price_scaler, best_stock_name, stock_data[best_stock_name]['Timestamp']
)
if predictions is not None:
    print(f"Best Model Test Adj R²: {test_adj_r2:.6f}")
    print(f"Predictions (first 5): {predictions[:5]}")
    print(f"Actuals (first 5): {actuals[:5]}")
    print(f"Test timestamps (first 5): {test_timestamps[:5]}")
else:
    print(f"Failed to generate test set predictions for {best_model_name}.")

Best Model Test Adj R²: 0.823853
Predictions (first 5): [5.2110796 5.2110796 5.2104216 5.210458  5.2099924]
Actuals (first 5): [5.21042576 5.21016265 5.21006215 5.21002376 5.2100091 ]
Test timestamps (first 5): 6350   2025-05-30 06:08:00+00:00
6351   2025-05-30 06:09:00+00:00
6352   2025-05-30 06:10:00+00:00
6353   2025-05-30 06:11:00+00:00
6354   2025-05-30 06:12:00+00:00
Name: Timestamp, dtype: datetime64[ns, UTC]


In [31]:
# Step 8: Generate Predictions for All Valid Models on Test Set
print("\nStep 8: Generating test set predictions for all models with Test Adj R² > 0.8...")
prediction_results = {}
for stock_name in stock_results.keys():
    valid_models = {}
    for model_name, result in stock_results[stock_name].items():
        if result.get('test_adj_r2', 0) > 0.65 and result.get('overfitting_gap', float('inf')) <= 0.2:
            valid_models[model_name] = result['best_model']
            print(f"Model '{model_name}' for {stock_name} meets criteria (Test Adj R² > 0.8, Overfitting Gap <= 0.2)")

    if not valid_models:
        print(f"No valid models found for stock {stock_name}")
        continue

    predictions_df = pd.DataFrame()
    test_X = stock_X_deep[stock_name][-150:] if any(m in ['cnn_lstm', 'conv1d_lstm', 'bilstm', 'stacked_lstm', 'gru', 'transformer', 'tcn'] for m in valid_models) else stock_X_ml[stock_name][-150:]
    test_y = stock_y[stock_name][-150:]
    test_timestamps = stock_data[stock_name]['Timestamp'].iloc[-150:]

    for model_name, model in valid_models.items():
        print(f"\nGenerating test set predictions for {model_name} in {stock_name}...")
        predictions, actuals, _, test_adj_r2 = evaluate_test_set(
            model, model_name, test_X, test_y, price_scaler, stock_name, stock_data[stock_name]['Timestamp']
        )
        if predictions is not None:
            predictions_df[model_name] = predictions
            print(f"Model {model_name} - Test Adj R²: {test_adj_r2:.6f}")

    predictions_df['Actual'] = price_scaler.inverse_transform(test_y.reshape(-1, 1)).flatten()
    predictions_df['Timestamp'] = test_timestamps
    prediction_results[stock_name] = predictions_df

for stock_name, predictions_df in prediction_results.items():
    print(f"\nTest Set Predictions for {stock_name}:")
    print(predictions_df.head())
    filename = f'{stock_name}_test_predictions.csv'
    predictions_df.to_csv(filename, index=False)
    print(f"\nDownloading {filename}...")
    files.download(filename)


Step 8: Generating test set predictions for all models with Test Adj R² > 0.8...
Model 'bilstm' for H78.SI meets criteria (Test Adj R² > 0.8, Overfitting Gap <= 0.2)
Model 'lightgbm' for H78.SI meets criteria (Test Adj R² > 0.8, Overfitting Gap <= 0.2)
Model 'xgboost' for H78.SI meets criteria (Test Adj R² > 0.8, Overfitting Gap <= 0.2)

Generating test set predictions for bilstm in H78.SI...
Model bilstm - Test Adj R²: 0.719202

Generating test set predictions for lightgbm in H78.SI...
Model lightgbm - Test Adj R²: 0.820278

Generating test set predictions for xgboost in H78.SI...
Model xgboost - Test Adj R²: 0.823853

Test Set Predictions for H78.SI:
     bilstm  lightgbm   xgboost    Actual Timestamp
0  5.215041  5.212834  5.211080  5.210426       NaT
1  5.211284  5.214576  5.211080  5.210163       NaT
2  5.210679  5.214782  5.210422  5.210062       NaT
3  5.211253  5.214061  5.210458  5.210024       NaT
4  5.211316  5.213786  5.209992  5.210009       NaT



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>